# Anime Action Scene — 3D Champ Inference
**Champ (3D SMPL Guidance) + Anything V5**

2D baseline과 비교 실험용. DWPose + MiDaS Depth + Normal 3가지 guidance 사용.

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 또는 A100 GPU** 선택 후 시작  
> ⚠️ 셀 2 실행 후 런타임 자동 재시작 → **셀 0, 1 다시 실행 후 셀 3으로**

In [ ]:
# ── 0. GPU 확인 ───────────────────────────────────────────────────────────────
import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── 1. 레포 클론 ───────────────────────────────────────────────────────────────
import os

if not os.path.exists('/content/anime-action-scene'):
    !git clone https://github.com/goldkangsan/anime-action-scene.git /content/anime-action-scene

%cd /content/anime-action-scene

# Champ 서브모듈 초기화 (champ/ 폴더 채워넣기)
!git submodule update --init --recursive
!ls champ/

In [ ]:
# ── 2. 의존성 설치 (처음 한 번만) ─────────────────────────────────────────────
# ⚠️ 실행 후 런타임 자동 재시작 → 재시작 후 셀 0, 1 실행하고 셀 3으로!

!pip install -q --upgrade pip

# 검증된 버전 조합 (huggingface_hub 0.23+ 에서 cached_download 삭제됨)
!pip install -q \
    "diffusers==0.24.0" \
    "transformers>=4.36.0,<4.40.0" \
    "huggingface_hub>=0.19.0,<0.23.0" \
    "accelerate"

!pip install -q \
    omegaconf einops \
    "controlnet-aux" \
    "onnxruntime-gpu" \
    imageio "imageio[ffmpeg]" av \
    mediapipe \
    gradio

# xformers: CUDA 버전 자동 감지 (cu121 하드코딩 제거)
!pip install -q xformers

print('\n✅ 설치 완료 — 런타임 자동 재시작 중...')
import os; os.kill(os.getpid(), 9)

In [ ]:
# ── 3. import 테스트 ───────────────────────────────────────────────────────────
import os, sys, warnings
warnings.filterwarnings('ignore')

os.chdir('/content/anime-action-scene')
sys.path.insert(0, '/content/anime-action-scene')
sys.path.insert(0, '/content/anime-action-scene/champ')  # Champ 모델 import용

# huggingface_hub 호환성 패치 (안전망)
import huggingface_hub as _hfhub
if not hasattr(_hfhub, 'cached_download'):
    _hfhub.cached_download = _hfhub.hf_hub_download
    sys.modules['huggingface_hub'].cached_download = _hfhub.hf_hub_download
    print('✅ huggingface_hub 호환성 패치 적용')

import diffusers, transformers
print(f'diffusers      : {diffusers.__version__}')
print(f'transformers   : {transformers.__version__}')
print(f'huggingface_hub: {_hfhub.__version__}')

# Champ 모델 import 확인
from models.unet_2d_condition import UNet2DConditionModel
from models.unet_3d import UNet3DConditionModel
from models.guidance_encoder import GuidanceEncoder
from models.champ_model import ChampModel
from pipelines.pipeline_aggregation import MultiGuidance2LongVideoPipeline

# AnimateAnyone src import 확인
from src.dwpose import DWposeDetector
from src.utils.util import get_fps, read_frames

print('\n✅ 모든 import 성공!')

In [ ]:
# ── 4. 가중치 다운로드 (~8GB, 처음 한 번만) ───────────────────────────────────
import os
os.chdir('/content/anime-action-scene')

# SD 1.5 + AnimateAnyone 공통 가중치
!python tools/download_weights.py

# Anything V5 (애니 스타일)
!python tools/download_weights.py --anime

# Champ 전용 가중치 (guidance encoder 4개 + denoising/reference unet)
!python tools/download_weights.py --champ

In [ ]:
# ── 5. 참조 이미지 업로드 ─────────────────────────────────────────────────────
import os, shutil
from google.colab import files
from pathlib import Path

os.chdir('/content/anime-action-scene')
Path('inputs').mkdir(exist_ok=True)

print('📁 애니 캐릭터 이미지를 업로드하세요 (jpg/png)')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, 'inputs/ref.png')
    print(f'  → inputs/ref.png 저장 완료')

In [ ]:
# ── 6. 액션 영상 업로드 ───────────────────────────────────────────────────────
import os, shutil
from google.colab import files
from pathlib import Path

os.chdir('/content/anime-action-scene')
Path('inputs').mkdir(exist_ok=True)

print('🎬 액션 영상을 업로드하세요 (mp4)')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, 'inputs/driving.mp4')
    print(f'  → inputs/driving.mp4 저장 완료')

In [ ]:
# ── 7. 업로드 확인 ─────────────────────────────────────────────────────────────
import os
os.chdir('/content/anime-action-scene')
from PIL import Image
import cv2
from IPython.display import display

ref = Image.open('inputs/ref.png')
print(f'참조 이미지: {ref.size}')
display(ref.resize((200, int(200 * ref.height / ref.width))))

cap = cv2.VideoCapture('inputs/driving.mp4')
fps  = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'드라이빙 영상: {total}프레임 @ {fps:.1f}fps')

In [ ]:
# ── 8. 전처리 — DWPose + Depth + Normal 추출 ──────────────────────────────────
# Champ 입력용 guidance 맵 생성 (MiDaS depth, Sobel normal, DWPose skeleton)
import os
os.chdir('/content/anime-action-scene')
from pathlib import Path
Path('outputs/guidance').mkdir(parents=True, exist_ok=True)

!python scripts/preprocess_smpl.py \
    --video inputs/driving.mp4 \
    --out outputs/guidance \
    --max_frames 32 \
    --width 384 \
    --height 512 \
    --device cuda

# 결과 확인
import glob
for folder in ['dwpose', 'depth', 'normal']:
    frames = glob.glob(f'outputs/guidance/{folder}/*.png')
    print(f'  {folder}: {len(frames)}장')

In [ ]:
# ── 9-A. Champ Inference — SD 1.5 (비교용 baseline) ───────────────────────────
import os, sys
os.chdir('/content/anime-action-scene/champ')
sys.path.insert(0, '/content/anime-action-scene/champ')

!python inference.py \
    --config ../configs/champ/colab_t4.yaml

os.chdir('/content/anime-action-scene')

In [ ]:
# ── 9-B. Champ Inference — Anything V5 (애니 특화) ────────────────────────────
import os, sys
os.chdir('/content/anime-action-scene/champ')
sys.path.insert(0, '/content/anime-action-scene/champ')

!python inference.py \
    --config ../configs/champ/colab_t4_anime.yaml

os.chdir('/content/anime-action-scene')

In [ ]:
# ── 10. 결과 확인 ──────────────────────────────────────────────────────────────
import os, glob
os.chdir('/content/anime-action-scene')
from IPython.display import Video, display

results = sorted(glob.glob('champ/results/**/*.mp4', recursive=True))
print(f'생성된 영상 {len(results)}개:')
for r in results:
    print(f'  {r}')

if results:
    print('\n▶ 최신 결과 미리보기:')
    display(Video(results[-1], width=512, embed=True))

In [ ]:
# ── 11. 결과 다운로드 ──────────────────────────────────────────────────────────
import os
os.chdir('/content/anime-action-scene')

!zip -r champ_results.zip champ/results/
from google.colab import files
files.download('champ_results.zip')
print('✅ champ_results.zip 다운로드 시작')